### Import

In [1]:
import math
import re
import requests
import chromadb
from pypdf import PdfReader
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker


### Inisialisasi Model Embeddings

In [2]:
embeddings = HuggingFaceEmbeddings(model_name="nlpaueb/legal-bert-base-uncased")

c:\Users\racha\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


No sentence-transformers model found with name nlpaueb/legal-bert-base-uncased. Creating a new one with mean pooling.


In [2]:
embeddingss = HuggingFaceEmbeddings(model_name="Qwen/Qwen3-Embedding-0.6B")

c:\Users\racha\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Fungsi Load & Pra Pemrosesan Pertor

#### Opsi 1

In [3]:
def preprocess_1(file_path: str) -> str:
    reader = PdfReader(file_path)
    raw_text = ""

    # 1. Ekstraksi PDF
    for _, page in enumerate(reader.pages, start=1):
        raw_text += page.extract_text() + "\n"

    text = raw_text

    # 2. Hilangkan header/footer standar UB
    text = re.sub(
        r"PEDOMAN PENDIDIKAN UNIVERSITAS BRAWIJAYA\s*\d+\/?\d*",
        "",
        text,
        flags=re.IGNORECASE
    )
    text = re.sub(
        r"Universitas Brawijaya\s*Rektor.*",
        "",
        text,
        flags=re.IGNORECASE
    )

    # 3. Hilangkan nomor halaman yang berdiri sendiri
    text = re.sub(r"\n\s*\d{1,3}\s*\n", "\n", text)

    # 4. Normalisasi newline > 3 menjadi \n\n
    text = re.sub(r"\n{3,}", "\n\n", text)

    # 5. Gabungkan baris patah TAPI jangan hilangkan paragraf
    #    Kondisi aman:
    #    - baris sebelumnya tidak diakhiri titik/dua titik/tanda tutup
    #    - baris berikutnya huruf kecil (lanjutan kalimat)
    text = re.sub(
        r"(?<![\.:\)])\n(?=[a-z])",
        " ",
        text
    )

    # 6. Hilangkan spasi berlebih
    text = re.sub(r"[ \t]{2,}", " ", text)

    # 7. Trim setiap baris
    text = "\n".join(line.strip() for line in text.splitlines())

    # 8. Hilangkan trailing newline berlebih
    final = text.strip()

    return final

#### Opsi 2

In [3]:
def preprocess_2(file_path: str) -> str:
    reader = PdfReader(file_path)
    raw_text = ""

    # 1. Ekstraksi PDF
    for _, page in enumerate(reader.pages, start=1):
        raw_text += page.extract_text() + "\n"

    text = raw_text

    # NOTE: Untuk dokumen Peraturan Rektor / legal format, header-footer tidak ada → rule berikut sengaja di-nonaktifkan
    # Jika nanti ada dokumen Pedoman Pendidikan (seperti dokumen 1), rule ini bisa diaktifkan:
    #
    # text = re.sub(
    #     r"PEDOMAN PENDIDIKAN UNIVERSITAS BRAWIJAYA\s*\d+\/?\d*",
    #     "",
    #     text,
    #     flags=re.IGNORECASE
    # )
    # text = re.sub(
    #     r"Universitas Brawijaya\s*Rektor.*",
    #     "",
    #     text,
    #     flags=re.IGNORECASE
    # )

    # 3. Hilangkan nomor halaman yang berdiri sendiri (versi aman)
    # HANYA hapus jika satu baris hanya berisi angka atau format "- 5 -"
    text = re.sub(
        r"(?m)^\s*[-–—]*\s*\d{1,3}\s*[-–—]*\s*$",
        "",
        text
    )

    # 4. Normalisasi newline > 3 menjadi \n\n
    text = re.sub(r"\n{3,}", "\n\n", text)

    # 5. Gabungkan baris patah TAPI jangan gabungkan item daftar, Pasal, BAB, angka hukum, dll.
    text = re.sub(
        r"(?<![\.\:\;\?\)])\n(?!\s*(BAB|Pasal|\d+\.|\(|- ))(?=[a-z])",
        " ",
        text
    )

    # 6. Hilangkan spasi tab/berlebih
    text = re.sub(r"[ \t]{2,}", " ", text)

    # 7. Trim setiap baris
    text = "\n".join(line.strip() for line in text.splitlines())

    # 8. Hilangkan trailing newline berlebih
    final = text.strip()

    return final


### Fungsi Fixed Chunking

In [4]:
def fixed_chunking(text: str,
                   chunk_size: int = 1000,
                   chunk_overlap: int = 200) -> list:
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        add_start_index=True,
    )
    
    chunks = text_splitter.split_text(text)
    docs = [Document(page_content=c) for c in chunks]
    return docs


### Inisialisasi ChromaDB

In [3]:
# Buat client manual dulu
client = chromadb.PersistentClient(path="./db_pertor_for_legal-bert")

# Buat collection dengan cosine similarity
collection = client.get_or_create_collection(
    name="fixed_chunk_cosine",
    configuration={
        "hnsw": {
            "space": "cosine",         # <- metric similarity
        }
    }
)

# Hubungkan ke LangChain
vector_store = Chroma(
    client=client,
    collection_name="fixed_chunk_cosine",
    embedding_function=embeddingss,
)


##### Pengecekan jika sudah ada data

In [6]:
# data = vector_store._collection.get(include=["embeddings", "documents"], limit=1)
# data = vector_store._collection.get( limit=111)
data= vector_store._collection.get(
    limit=1,
    offset=100
)
print (data)

existing_count = vector_store._collection.count()

print(existing_count)


{'ids': ['b21e1ff9-c8bb-4dc8-b080-2c4d0f280c21'], 'embeddings': None, 'documents': ['Komunikasi, Ilmu Administrasi Publik, Ilmu Administrasi Bisnis,\nAgroekoteknologi, Agrobisnis, Teknik Informatika, Teknologi Industri\nPertanian, Ilmu Keperawatan, dan Agrobisnis Perikanan.\nBerdasarkan Surat Keputusan Rektor No. 516/SK/2011 tangga l 27\nOktober 2011 telah dibentuk Program Teknologi Informasi dan Ilmu\nKomputer Universitas Brawijaya dengan 4 Program Studi S1 yaitu Program\nStudi Teknik Informatika, Program Studi Ilmu Komputer, Program Studi\nTeknik Komputer dan Program Studi Sistem Infor masi. Dimana sebelumnya\nProgram studi Teknik Informatika, Teknik Komputer dan Sistem informasi merupakan Program Studi yang berada di bawah Fakultas Teknik, sedangkan\nProgram Studi Ilmu Komputer berada di bawah Fakultas MIPA.\nPada tahun 2014, berdasarkan K eputusan Menteri Pendidikan dan\nKebudayaan RI No. 595/E/O/2014, Tanggal 17 Oktober 2014 Universitas'], 'uris': None, 'included': ['metadatas', '

## Penggunaan

#### Load & Preprocessing

In [ ]:
# Example
# file_path = r"D:\document\PERATURAN REKTOR UNIVERSITAS BRAWIJAYA NOMOR 89 TAHUN 2023 TENTANG TATA NASKAH DINAS.pdf"
file_path = r"your_doc_path"

# clean_text = preprocess_1(file_path)
clean_text = preprocess_2(file_path)

#### Chunking

In [8]:
chunks = fixed_chunking(clean_text)

In [30]:
import os

chunk_file_path = os.path.join( "chunks_docs_4.txt")

# Simpan semua chunk ke satu file

with open(chunk_file_path, "w", encoding="utf-8") as f:
    for i, chunk in enumerate(chunks, start=1):
        f.write(f"=== Chunk ke-{i} ===\n\n")
        f.write(chunk.page_content.strip() + "\n\n\n")

print(f"✔ Semua {len(chunks)} chunk disimpan dalam 1 file:")
print(f"  {chunk_file_path}")

✔ Semua 94 chunk disimpan dalam 1 file:
  chunks_docs_4.txt


#### Menyimpan ke chormadb

In [9]:
print(len(chunks))

vector_store.add_documents(chunks)

94


['35d1bdc5-575d-45e3-b46c-21a27a8a4342',
 '3959a33a-04c0-42db-95ce-799c311ee01e',
 '565dcd64-2af7-477e-869a-5e07d7656d8e',
 'a53bb772-e3ca-4b92-af7a-4d6fea0055eb',
 '4d7866db-6452-494c-9c12-68d78439aa37',
 '40179742-c2c9-41bc-be0f-be2361c71d19',
 '05ad1952-f73b-4a30-90c4-060f5934ad3d',
 'a9abd367-0bd7-4c4d-8670-a1092772fa29',
 '3b0bafba-039a-4ab9-8469-18ee677596bf',
 'e65b3fb1-d983-4306-b069-ddedff5d8481',
 'fe7394bb-8d05-428b-993c-496dea1c8d12',
 '11538b19-946f-410f-827b-2b628457616e',
 '39b68db9-147f-4b57-bfc1-8711bec81caf',
 '5d5f13e1-5e0f-491b-bf4a-13ec35829650',
 '294db5b0-fd3e-43ee-a222-ba116219a9f1',
 '2d7292b1-5035-4868-9209-1d2d317d8118',
 '7670e9e2-5d25-4cc0-baf6-d32f4ceb3c99',
 '7388c500-6ce1-4871-9ab1-cbe181877f2e',
 'a72b95b0-f114-419f-8b9c-cc78f5e934c5',
 '850c6430-4cbc-44c3-a5a8-a38f55379f53',
 '3dcfad84-4035-4fab-a746-0bc9482fd4a8',
 '2b934ab9-a2e2-425d-ae92-08379b3589e9',
 '6020783d-a7ff-4204-8a24-3a076f5fc06d',
 '44b0dbb7-8072-464e-be92-0dc7e0afdf16',
 'a313c241-69d5-

#### Pengecekan

In [10]:
# data = vector_store._collection.get(limit=1)
existing_count = vector_store._collection.count()

print(existing_count)

# print(vector_store._collection_configuration)


977


### Query Retrieval

In [4]:
# Apa pengertian dari kurikulum?
# Apa saja syarat yudisium?
question = input("Masukkan pertanyaan: ")

retrieved_docs = vector_store.similarity_search(question, k=5)
# retrieved_docs = vector_store.similarity_search_by_vector(embedding=embeddings.embed_query(question), k=5)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Chunk {i} ---\n {doc.page_content} [{doc.metadata}]")



--- Chunk 1 ---
 Mutu (LP3M) Universitas Brawijaya.
Kurikulum berfungsi sebagai instrumen untuk membentuk pola pikir ilmiah, keahlian, dan kepribadian mahasiswa. Oleh karena itu kurikulum harus mendorong pemenuhan capaian pembelajaran program studi yang dibutuhkan berupa pengetahuan dan pemahaman, keahlian kognitif, keahlian khusus (termasuk keahlian praktis atau profesional), keahlian yang dapat ditransfer, kebutuhan untuk pekerjaan dan atau studi lanjut, serta pengembangan kepribadian. [{}]

--- Chunk 2 ---
 Kurikulum di Universitas Brawijaya merupakan landasan utama penyelenggaraan pendidikan akademik, profesi, spesialis dan vokasi menuju pencapaian hasil belajar sesuai dengan standar lu lusan
Universitas Brawijaya. Kurikulum merupakan seperangkat rencana dan peraturan mengenai isi atau bahan kajian dan materi pembelajaran, serta cara penyampaian maupun cara penilaian untuk menjamin tercapainya kompetensi lulusan. Oleh karenanya keberadaan kurikulum dijadikan sebagai acuan pokok ba

In [32]:
question = "Pengertian kode etik mahasiswa Universitas Brawijaya ada di Peraturan Nomor berapa?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "topk_chunks.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: topk_chunks.txt


##### Query/Pertanyaan 1 (Q1)

In [6]:
question = "Berapa jatah SKS yang bisa diambil kalau IP semester lalu di atas 3,00?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Muflih/legal-fixed-q1.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Muflih/legal-fixed-q1.txt


##### Query/Pertanyaan 2 (Q2)

In [7]:
question = "Gimana prosedur mengurus ijazah yang hilang atau rusak?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Muflih/legal-fixed-q2.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Muflih/legal-fixed-q2.txt


##### Query/Pertanyaan 3 (Q3)

In [8]:
question = "Apa sanksi akademik kalau mahasiswa ketahuan curang saat ujian?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Muflih/legal-fixed-q3.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Muflih/legal-fixed-q3.txt


##### Query/Pertanyaan 4 (Q4)

In [9]:
question = "Apa saja tugas dan fungsi dari UPT Perpustakaan?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Muflih/legal-fixed-q4.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Muflih/legal-fixed-q4.txt


##### Query/Pertanyaan 5 (Q5)

In [10]:
question = "Apa yang dimaksud dengan Naskah Dinas dan apa saja jenisnya?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Muflih/legal-fixed-q5.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Muflih/legal-fixed-q5.txt


##### Query/Pertanyaan 6 (Q6)

In [36]:
question = "Apa saja tugas dari Direktorat Inovasi dan Kawasan Sains dan Teknologi?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Yusuf/legal-fixed-6.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Yusuf/legal-fixed-6.txt


##### Query/Pertanyaan 7 (Q7)

In [37]:
question = "Maksimal mata kuliah di semester antara berapa SKS?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Yusuf/legal-fixed-7.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Yusuf/legal-fixed-7.txt


##### Query/Pertanyaan 8 (Q8)

In [38]:
question = "Apa saja syarat yudisium?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Yusuf/legal-fixed-8.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Yusuf/legal-fixed-8.txt


##### Query/Pertanyaan 9 (Q9)

In [39]:
question = "Apa isi Pasal 31 pada dokumen pertor tentang tata naskah dinas?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Yusuf/legal-fixed-9.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Yusuf/legal-fixed-9.txt


##### Query/Pertanyaan 10 (Q10)

In [40]:
question = "Apa itu program pendidikan akademik doktor?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Yusuf/legal-fixed-10.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Yusuf/legal-fixed-10.txt


##### Query/Pertanyaan 11 (Q11)

In [41]:
question = "Saya memiliki nilai TOEFL 400 apakah bisa mendaftar program fast track?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Yusuf/legal-fixed-11.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Yusuf/legal-fixed-11.txt


##### Query/Pertanyaan 12 (Q12)

In [42]:
question = "Dana abadi digunakan untuk apa? Apakah ada digunakan untuk beasiswa mahasiswa?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Yusuf/legal-fixed-12.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Yusuf/legal-fixed-12.txt


##### Query/Pertanyaan 13 (Q13)

In [43]:
question = "Pengertian SKPI"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Nadhira/legal-fixed-q13.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Nadhira/legal-fixed-q13.txt


##### Query/Pertanyaan 14 (Q14)

In [44]:
question = "Kapan SKPI dikeluarkan?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Nadhira/legal-fixed-q14.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Nadhira/legal-fixed-q14.txt


##### Query/Pertanyaan 15 (Q15)

In [45]:
question = "Apa saja isi dari SKPI?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Nadhira/legal-fixed-q15.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Nadhira/legal-fixed-q15.txt


##### Query/Pertanyaan 16 (Q16)

In [46]:
question = "Bagaimana jika mahasiswa memerlukan ijazah asli sebelum wisuda?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Nadhira/legal-fixed-q16.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Nadhira/legal-fixed-q16.txt


##### Query/Pertanyaan 17 (Q17)

In [47]:
question = "Apakah bisa dilakukan perbaikan ijazah?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Nadhira/legal-fixed-q17.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Nadhira/legal-fixed-q17.txt


##### Query/Pertanyaan 18 (Q18)

In [14]:
question = "Materi muatan peraturan dekan berisi materi yang mengatur tentang?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Jihaan/legal-fixed-q18.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Jihaan/legal-fixed-q18.txt


##### Query/Pertanyaan 19 (Q19)

In [5]:
question = "Apa pengertian dari kurikulum?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Jihaan/legal-fixed-q19.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Jihaan/legal-fixed-q19.txt


##### Query/Pertanyaan 20 (Q20)

In [6]:
question = "Apa saja prinsip penerbitan Ijazah, Transkrip Nilai, dan SKPI?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Jihaan/legal-fixed-q20.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Jihaan/legal-fixed-q20.txt


##### Query/Pertanyaan 21 (Q21)

In [7]:
question = "Apa saja alasan agar Ijazah, Transkrip Nilai, SKPI, dan Sertifikat Profesi dapat diterbitkan ulang?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Jihaan/legal-fixed-q21.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Jihaan/legal-fixed-q21.txt


##### Query/Pertanyaan 22 (Q22)

In [8]:
question = "Pengertian dari SAF?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Jihaan/legal-fixed-q22.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Jihaan/legal-fixed-q22.txt


##### Query/Pertanyaan 23 (Q23)

In [9]:
question = "Apa saja tugas Direktorat Administrasi dan Layanan Akademik?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Jihaan/legal-fixed-q23.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Jihaan/legal-fixed-q23.txt


##### Query/Pertanyaan 24 (Q24)

In [10]:
question = "Fungsi dari Direktorat Inovasi dan Pengembangan Pendidikan?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Jihaan/legal-fixed-q24.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Jihaan/legal-fixed-q24.txt


##### Query/Pertanyaan 25 (Q25)

In [11]:
question = "Apabila mahasiswa masih memiliki file elektronik dari Ijazah, Transkrip Nilai, SKPI, dan Sertifikat Profesi maka pencetakan ulang dapat dilakukan dimana?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Jihaan/legal-fixed-q25.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Jihaan/legal-fixed-q25.txt


##### Query/Pertanyaan 26 (Q26)

In [11]:
question = "Apa saja pilihan kegiatan Merdeka Belajar di luar kampus yang bisa diambil?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Muflih/legal-fixed-q26.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Muflih/legal-fixed-q26.txt


##### Query/Pertanyaan 27 (Q27)

In [12]:
question = "Apa saja informasi yang wajib dicantumkan di dalam SKPI?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/Muflih/legal-fixed-q27.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/Muflih/legal-fixed-q27.txt


##### Query/Pertanyaan 28 (Q28)

In [12]:
question = "Apa itu program studi?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/There/legal-fixed-q28.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/There/legal-fixed-q28.txt


##### Query/Pertanyaan 29 (Q29)

In [ ]:
question = "Apa itu departemen?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/There/legal-fixed-q29.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/There/legal-fixed-q29.txt


: 

##### Query/Pertanyaan 30 (Q30)

In [33]:
question = "Berapa lama MBKM boleh dilaksanakan?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/There/legal-fixed-q30.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/There/legal-fixed-q30.txt


##### Query/Pertanyaan 31 (Q31)

In [34]:
question = "Perbedaan model interaksi asinkron dan sinkronisasi"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/There/legal-fixed-q31.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/There/legal-fixed-q31.txt


##### Query/Pertanyaan 32 (Q32)

In [35]:
question = "Peraturan SAF dibentuk untuk apa?"

retrieved_docs = vector_store.similarity_search(question, k=5)

output_path = "Jawaban/There/legal-fixed-q32.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"Pertanyaan: {question}\n")
    f.write("="*60 + "\n\n")

    for i, doc in enumerate(retrieved_docs, 1):
        f.write(f"--- Chunk {i} ---\n")
        f.write(doc.page_content.strip() + "\n\n")

print(f"Berhasil menyimpan top-k chunk ke: {output_path}")

Berhasil menyimpan top-k chunk ke: Jawaban/There/legal-fixed-q32.txt


### Pengujian

In [ ]:
def precision_at_k(relevance_labels, k=None):
    if k is None:
        k = len(relevance_labels)
    rel = relevance_labels[:k]
    return sum(rel) / k if k > 0 else 0

def reciprocal_rank(relevance_labels):
    try:
        rank = relevance_labels.index(1) + 1
        return 1 / rank
    except ValueError:
        return 0

def mean_reciprocal_rank(all_queries_labels):
    if len(all_queries_labels) == 0:
        return 0

    rr_values = [reciprocal_rank(labels) for labels in all_queries_labels]
    return sum(rr_values) / len(rr_values)

def average_precision(relevance_labels):
    rel = relevance_labels
    total_rel = sum(rel)

    if total_rel == 0:
        return 0

    precision_sum = 0
    for i in range(len(rel)):
        if rel[i] == 1:
            precision_at_i = sum(rel[:i+1]) / (i+1)
            precision_sum += precision_at_i

    return precision_sum / total_rel

def mean_average_precision(all_queries_labels):
    if len(all_queries_labels) == 0:
        return 0

    ap_values = [average_precision(labels) for labels in all_queries_labels]
    return sum(ap_values) / len(ap_values)

def ndcg_at_k(relevance_labels, k=None):
    if k is None:
        k = len(relevance_labels)

    rel = relevance_labels[:k]

    # DCG
    dcg = 0
    for i in range(k):
        dcg += (2**rel[i] - 1) / math.log2(i + 2)

    # IDCG (label ideal, urut dari relevan tertinggi)
    ideal = sorted(rel, reverse=True)
    idcg = 0
    for i in range(k):
        idcg += (2**ideal[i] - 1) / math.log2(i + 2)

    return dcg / idcg if idcg > 0 else 0



In [ ]:
labels = [1, 0, 1, 0, 1]

print("Precision@k:", precision_at_k(labels))
print("NDCG@k:", ndcg_at_k(labels))

queries_labels = [
    [1, 0, 1, 0, 1],   # Query 1
    [0, 0, 1, 0, 1],   # Query 2
    [0, 1, 0, 0, 1],   # Query 3
    [0, 0, 0, 1, 1],   # Query 4
    [1, 1, 0, 0, 0],   # Query 5
    [0, 1, 1, 0, 0]    # Query 5
]

# Hitung MRR & MAP untuk 6 query
print("MRR:", mean_reciprocal_rank(queries_labels))
map_score = mean_average_precision(queries_labels)
print("MAP:", map_score)

Precision@k: 0.6
NDCG@k: 0.8854598815714874
MRR: 0.5972222222222222
MAP: 0.5800925925925926
